# Todo list
- [ ] Shows masks -> semantic segmentation
- [ ] Choose better variable names
- [ ] learn to use logger debug etc

# NUSCENES detection task

## Context

Autonomous driving systems rely on **perception**, meaning the part of the system that turns raw sensor data into an understanding of the surroundings. A core perception task is **3D object detection**, which means predicting what is present (for example car, pedestrian, bicycle) and where it is in 3D space using a **3D bounding box**. A **3D bounding box** describes an object’s position, size, and orientation in space.

The **nuScenes** dataset provides synchronized data from multiple sensors, including cameras, LiDAR, and radar, which makes it possible to compare sensors fairly. **Sensor fusion** means combining information from more than one sensor in a single system. The plan is to start from a strong LiDAR only detector, then measure what changes when adding camera information (visual cues) and radar information (motion cues), and show where these additions help most, for example by object type and distance.

Why start with LiDAR only? LiDAR directly measures 3D distances, so it provides a strong and stable baseline for 3D bounding boxes. It also keeps the first setup simpler, which makes evaluation and debugging easier. With a reliable baseline, improvements from adding cameras or radar can be measured clearly.

## The problem

Using more sensors increases system complexity, so it is important to measure what each sensor actually contributes. The problem I address is: how much does each modality (camera, LiDAR, radar) improve 3D detection quality on nuScenes, and which types of errors are reduced? I will answer this with a controlled **ablation study**, meaning a comparison where only one factor changes at a time (here, the sensor inputs), using standard nuScenes evaluation metrics.

Core question:  
* How much performance gain comes from adding camera and or radar compared to LiDAR only?

How it will be tested:  
* Compare three variants: LiDAR only, camera plus LiDAR, camera plus LiDAR plus radar  
* Evaluate with official nuScenes metrics: NDS (overall score), mAP (detection accuracy), and AVE (velocity error)

What I will deliver:  
* A results table comparing the three variants on the same metrics  
* A short error breakdown, for example by class and distance, to explain why gains happen

# The data
## Source and context of the dataset (nuScenes)

**nuScenes** is a public **multimodal autonomous-driving dataset** released by **Motional (nuTonomy)**. It contains **1,000 urban driving scenes** of about **20 seconds** each, recorded in **Boston** and **Singapore**.  
Each scene includes synchronized data from a full sensor suite: **6 cameras, 1 LiDAR, and 5 radars**, plus vehicle localization sensors such as GPS/IMU.  
nuScenes is widely used to develop and evaluate tasks like **3D object detection**, where the goal is to predict **3D bounding boxes** and **class labels** for road users.

For preliminary exploration, I will use **nuScenes-mini**, a small subset of nuScenes designed for quick tests. It allows faster debugging of the data pipeline, visualization, and evaluation before running full training on the complete dataset.

• Dataset overview: https://www.nuscenes.org/nuscenes  
• Download: https://www.nuscenes.org/download  
• 3D object detection benchmark: https://www.nuscenes.org/object-detection  
• Official devkit and evaluation tools: https://github.com/nutonomy/nuscenes-devkit  
• Dataset paper: https://openaccess.thecvf.com/content_CVPR_2020/papers/Caesar_nuScenes_A_Multimodal_Dataset_for_Autonomous_Driving_CVPR_2020_paper.pdf

## Database schema

**Source:** The following text and schema image were copied from the official nuScenes tutorial page https://github.com/nutonomy/nuscenes-devkit/blob/master/python-sdk/tutorials/nuscenes_tutorial.ipynb

1. `log` - Log information from which the data was extracted.
2. `scene` - 20 second snippet of a car's journey.
3. `sample` - An annotated snapshot of a scene at a particular timestamp.
4. `sample_data` - Data collected from a particular sensor.
5. `ego_pose` - Ego vehicle poses at a particular timestamp.
6. `sensor` - A specific sensor type.
7. `calibrated sensor` - Definition of a particular sensor as calibrated on a particular vehicle.
8. `instance` - Enumeration of all object instance we observed.
9. `category` - Taxonomy of object categories (e.g. vehicle, human).
10. `attribute` - Property of an instance that can change while the category remains the same.
11. `visibility` - Fraction of pixels visible in all the images collected from 6 different cameras.
12. `sample_annotation` - An annotated instance of an object within our interest.
13. `map` - Map data that is stored as binary semantic masks from a top-down view.

The database schema is visualized below. For more information see the [nuScenes schema](https://github.com/nutonomy/nuscenes-devkit/blob/master/docs/schema_nuscenes.md) page.
![](https://www.nuscenes.org/public/images/nuscenes-schema.svg)

# Setup

In [1]:
# !mkdir -p /data/sets/nuscenes  # Make the directory to store the nuScenes dataset in.

# !wget https://www.nuscenes.org/data/v1.0-mini.tgz  # Download the nuScenes mini split.

# !tar -xf v1.0-mini.tgz -C /data/sets/nuscenes  # Uncompress the nuScenes mini split.

# !pip install nuscenes-devkit &> /dev/null  # Install nuScenes.

## Import

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import logging

from IPython.display import display

logging.basicConfig(level=logging.INFO, format="%(message)s")
logger = logging.getLogger("notebook")

pd.set_option('display.max_columns', None)
# pd.set_option("display.max_colwidth", None)

In [3]:
import os
from nuscenes.nuscenes import NuScenes

DATAROOT: str= "../data/sets/nuscenes"
VERSION: str = "v1.0-mini"

assert os.path.isdir(DATAROOT), f"DATAROOT not found: {os.path.abspath(DATAROOT)}"

# Load the nuScenes tables. 
nusc = NuScenes(version=VERSION, dataroot=DATAROOT, verbose=True)

ModuleNotFoundError: No module named 'cv2'

## Utility functions

In [ ]:
# help(nusc.get)
# help(nusc.get_sample_data)
# help(nusc.render_sample)
# help(nusc.render_sample_data)
# help(nusc.render_pointcloud_in_image)
# help(nusc.list_scenes())

# Load and inspect nuScenes tables
For the full nuScenes dataset, they started with a large amount of training data and then hand-picked 84 logs, totaling about 15 hours of driving.

https://openaccess.thecvf.com/content_CVPR_2020/papers/Caesar_nuScenes_A_Multimodal_Dataset_for_Autonomous_Driving_CVPR_2020_paper.pdf

In [ ]:
# Dataset version (e.g., "v1.0-mini" = development subset)
print("version:", nusc.version)

# Number of logs (recording sessions collected by the ego vehicle)
print("number of logs:", len(nusc.log))

# Number of scenes (~20s segments extracted from logs)
print("number of scenes:", len(nusc.scene))

# List of available tables (nuScenes is organized as a relational-style database)
print("List of tables:", nusc.table_names)


## Database tables

### `Scene`
A scene is a short driving clip from one recording log, about 20 seconds long. It contains many time steps called samples with synchronized sensor data and labels.

scene → first_sample_token → sample (time step) → sample_data (sensor files) + anns (labels)

In [ ]:
# Prints a summary list of all scenes in the dataset (name, description, and basic info).
nusc.list_scenes()

In [ ]:
from pprint import pprint

# Select the first scene and print its metadata (e.g., name, description, log it belongs to, etc.).
scene0: dict = nusc.scene[0]
pprint(scene0, sort_dicts=False)

### `Sample`
* Inside a scene, there are many time steps called samples.
* A sample is a snapshot taken every 0.5 seconds, so two snapshots per second.
* Each sample gives the camera image, the lidar and radar data from that same moment.

In [ ]:
# --- First sample of the scene ---
# A scene is a time-ordered linked list of samples; a sample is one timestamp of synchronized sensor data.
sample0: dict = nusc.get("sample", scene0["first_sample_token"])

pprint(sample0, sort_dicts=False)

# I then load the second sample.
sample1 = nusc.get("sample", sample0["next"])

In [ ]:
# I compute the time difference between two consecutive samples.
# Timestamps are stored in microseconds, so I convert the result to seconds.
dt_sec = (sample1["timestamp"] - sample0["timestamp"]) / 1e6
print("\nTime between samples (sec):", dt_sec)

In [ ]:
# Render the first sample
nusc.render_sample(sample0["token"])

### `Sample data`
A sample_data is a single sensor measurement from one sensor.

* Radar sensors can detect objects about 200 to 300 meters away and estimate their speed using the Doppler effect. They work well in rain, fog, and low light, but their measurements are lower resolution and often noisy, so the detections can look sparse and hard to interpret.
* LiDAR sensors measure distance by sending out laser pulses and timing how long they take to return. This produces a 3D point cloud with accurate object shapes and positions, but it can be sparse at long range and may be affected by rain, fog, or snow.


"However, lidar data is sparse and the range is typically limited
to 50-150m."

In [ ]:
# Get sample_data 
cam_front_sd: dict = nusc.get("sample_data", sample0["data"]["CAM_FRONT"])
lidar_top_sd: dict = nusc.get("sample_data", sample0["data"]["LIDAR_TOP"])
radar_front_sd: dict = nusc.get("sample_data", sample0["data"]["RADAR_FRONT"])

pprint(cam_front_sd, sort_dicts=False)

In [ ]:
nusc.render_sample_data(cam_front_sd["token"])

In [ ]:
nusc.render_sample_data(lidar_top_sd["token"])

In [ ]:
nusc.render_sample_data(radar_front_sd["token"])

In [ ]:
nusc.render_pointcloud_in_image(
    sample_token=sample1["token"],
    pointsensor_channel="RADAR_FRONT",
    camera_channel="CAM_FRONT"
)

### `sample_annotation`
A **labeled object at one sample timestamp**. It stores the object’s **3D bounding box** (category, size, position and orientation) and connects to the **same physical object across frames** via `instance_token` (and the `prev` / `next` links along the track).

In [ ]:
import matplotlib.pyplot as plt

# show few annotations of the first sample
for i in range(3): 
    print(f'Annotation #: {i}')
    ann = nusc.get("sample_annotation", sample0["anns"][i])
    pprint(ann, sort_dicts=False)
    nusc.render_annotation(ann["token"])
    plt.show() 

### `Instance`
* instance: a unique tracked object in a scene (e.g., the same car or pedestrian over time).
* It links all corresponding sample_annotation records across timestamps via first/last_annotation_token.

In [ ]:
# --- I load and visualize one annotation from this sample ---
ann_idx: int = 7
ann_token: str = sample0["anns"][ann_idx]
ann: dict = nusc.get("sample_annotation", ann_token)

print(f"\n--- annotation {ann_idx} (sample_annotation) ---")
pprint(ann, sort_dicts=False)
nusc.render_annotation(ann["token"])
plt.show()

In [ ]:
# --- I inspect the corresponding instance (same object across time) ---
instance: dict = nusc.get("instance", ann["instance_token"])
instance_category: str = nusc.get("category", instance["category_token"])["name"]

print("\n--- instance (object identity across time) ---")
pprint(instance, sort_dicts=False)
print("\ninstance token:", instance["token"])
print("category:", instance_category)
print("#annotations in track:", instance["nbr_annotations"])

In [ ]:
# --- I follow the same instance track for up to 3 next annotations ---
ann_token: str = ann["next"]  # start from the next annotation after 'ann'

for step in range(1, 4):  # 1..3
    if not ann_token:
        print("\nNo next annotation for this instance (track ends here).")
        break

    ann_step: dict = nusc.get("sample_annotation", ann_token)

    print(f"\n--- next annotation in the same track (step {step}/3) ---")
    pprint(ann_step, sort_dicts=False)
    nusc.render_annotation(ann_step["token"])
    plt.show()

    ann_token = ann_step["next"]

### `category`

Defines the **semantic class label** (e.g., `vehicle.car`, `human.pedestrian.adult`) used to name and group objects.

- Stores a **hierarchical class name** in `name` (dot-separated taxonomy).
- Linked from annotations/instances via `category_token`.
- Used to **standardize labels** across the dataset.

`nusc.list_categories()` prints **per-category stats of 3D bounding-box sizes** for the selected split.

For each category it reports:
- **`n`**: number of `sample_annotation` boxes
- **`width / len / height`**: mean ± std box dimensions (meters)
- **`lw_aspect`**: mean ± std of `len / width`

In [ ]:
nusc.list_categories()

In [ ]:
# --- Go from a sample_annotation -> category record ---
ann: dict = nusc.get("sample_annotation", sample0["anns"][0])  # one labeled box at one timestamp

print("\n--- annotation (sample_annotation) ---")
pprint(ann, sort_dicts=False)

print("Annotation category_name:", ann["category_name"])

# category table uses tokens; convert name -> token, then fetch the record
cat_token: str = nusc.field2token("category", "name", ann["category_name"])[0]
cat: dict = nusc.get("category", cat_token)

print("\nCategory record (from category_name):")
pprint(cat, sort_dicts=False)

# --- Bonus: same category, accessed via the instance (track) ---
inst: dict = nusc.get("instance", ann["instance_token"])
cat2: dict = nusc.get("category", inst["category_token"])

print("\nCategory record (via instance -> category_token):")
pprint(cat2, sort_dicts=False)

### `attribute`

**Extra label** that refines an annotation’s meaning (e.g., `vehicle.moving`, `cycle.with_rider`).

- Stored as an `attribute` record (`name`, `description`).
- Linked from `sample_annotation` via `attribute_tokens` (0..N per annotation).

In [ ]:
nusc.list_attributes()

In [ ]:
# Code from https://github.com/nutonomy/nuscenes-devkit/blob/master/python-sdk/tutorials/nuscenes_tutorial.ipynb

my_instance = nusc.instance[27]
first_token = my_instance['first_annotation_token']
last_token = my_instance['last_annotation_token']
nbr_samples = my_instance['nbr_annotations']
current_token = first_token

i = 0
found_change = False
while current_token != last_token:
    current_ann = nusc.get('sample_annotation', current_token)
    current_attr = nusc.get('attribute', current_ann['attribute_tokens'][0])['name']

    if i == 0:
        pass
    elif current_attr != last_attr:
        print("Changed from `{}` to `{}` at timestamp {} out of {} annotated timestamps".format(last_attr, current_attr, i, nbr_samples))
        found_change = True

    next_token = current_ann['next']
    current_token = next_token
    last_attr = current_attr
    i += 1

### `visibility`

Indicates **how much of an object is visible** in the camera view (occlusion level).

- Stored as a `visibility` record (`token`, `level`, `description`).
- Referenced by `sample_annotation["visibility_token"]`.

In [ ]:
# There are for visibility categories
nusc.visibility

In [ ]:
# Pick any annotation (one box at one timestamp)
ann: dict = nusc.get("sample_annotation", sample0["anns"][0])

print("\n--- annotation (sample_annotation) ---")
pprint(ann,sort_dicts=False)
print("Annotation token:", ann["token"])
print("Category:", ann["category_name"])

# --- Visibility (occlusion / how much is visible) ---
vis: dict = nusc.get("visibility", ann["visibility_token"])
print("\nVisibility record:")
pprint(vis, sort_dicts=False)
nusc.render_annotation(ann["token"])

### `sensor`

The nuScenes dataset is collected using a full sensor suite comprising:
- 1 x LIDAR
- 5 x RADAR
- 6 x cameras

In [ ]:
nusc.sensor

Every `sample_data` entry records **which sensor** produced it (see the linked sensor’s **`channel`**).

In [ ]:
nusc.sample_data[10]

### `calibrated_sensor`

Describes the sensor’s **position on the vehicle** and its **direction/rotation**.

In [ ]:
# Get the front camera sample_data, then its calibrated_sensor record
sd = nusc.get("sample_data", sample0["data"]["CAM_FRONT"])
print("\n--- sample_data (CAM_FRONT) ---")
pprint(sd, sort_dicts=False)
cs = nusc.get("calibrated_sensor", sd["calibrated_sensor_token"])
print("\n--- calibrated_sensor (CAM_FRONT) ---")
pprint(cs, sort_dicts=False)

print("Sensor mounted position (translation, meters):", cs["translation"])
print("Sensor orientation (rotation quaternion [w,x,y,z]):", cs["rotation"])

### `ego_pose`

Describes **where the car is** and **which way it is facing** at a specific timestamp with respect to the global coordinate.

- Stored as an `ego_pose` record (`token`, `translation`, `rotation`, `timestamp`).
- Referenced by `sample_data["ego_pose_token"]`.

In [ ]:
ego0 = nusc.ego_pose[0]
pprint(ego0, sort_dicts=False)

### `log`

Stores **information about the recording session** (where and with which vehicle the data was collected).

- Stored as a `log` record (`token`, `logfile`, `vehicle`, `date_captured`, `location`).
- Referenced by `scene["log_token"]`.

In [ ]:
# 1) Pick one scene (first scene in the dataset)
scene0: dict = nusc.scene[5]
print("Scene name:", scene0["name"])

# 2) Get its log record (recording session info)
log0: dict = nusc.get("log", scene0["log_token"])

# 3) Show the raw dict (what's stored in the dataset)
pprint(log0, sort_dicts=False)

### `map`

In [ ]:
# TODO: illustrate map


## Load nuscenes table into dict of dataframes

In [ ]:
# Load nuscenes table into dict of dataframes
nusc_tables: dict[str, pd.DataFrame] = {table_name: pd.DataFrame(getattr(nusc, table_name)) for table_name in nusc.table_names}
print("List of dataframes:", list(nusc_tables.keys()))

In [ ]:
for t in list(nusc_tables.keys()):
    df = nusc_tables[t]
    print(f"\n{t}: {df.shape}")
    print(df.columns.tolist())

In [ ]:
for t in list(nusc_tables.keys()):
    print(f"\n--- {t} ---")
    display(nusc_tables[t].head())

# Build the annotation-level DataFrame

In [ ]:
df = nusc_tables['sample_annotation'].copy()
df.head()

In [ ]:
new_cols: dict[str, dict[str, str]] = { 
    "timestamp": {"token": "sample_token", 
                  "table_name": "sample", 
                  "column": "timestamp"},
    "scene_token": {"token": "sample_token", 
                    "table_name": "sample", 
                    "column": "scene_token"},
    "scene_name": {"token": "scene_token", 
                   "table_name": "scene", 
                   "column": "name"},
    "scene_description": {"token": "scene_token", 
                          "table_name": "scene", 
                          "column": "description"},
    "log_token": {"token": "scene_token", 
                  "table_name": "scene", 
                  "column": "log_token"},
    "location": {"token": "log_token", 
                 "table_name": "log", 
                 "column": "location"},
    "category_token": {"token": "instance_token", 
                       "table_name": "instance", 
                       "column": "category_token"},
    "category_description": {"token": "category_token", 
                            "table_name": "category", 
                            "column": "description"},
    } 
new_cols

In [ ]:
def map_token_to_data(nusc_tables: pd.DataFrame, field_name: str) -> pd.Series:
    return nusc_tables.set_index("token")[field_name]

for key, value in new_cols.items():
    df[key] = df[value["token"]].map(
        map_token_to_data(
            nusc_tables=nusc_tables[value["table_name"]],
            field_name=value["column"],
        )
    )

df.head()

### Add attribute

In [ ]:
attr_token_to_name: dict[str, str] = (
    nusc_tables["attribute"]
    .set_index("token")["name"]
    .to_dict()
)

pprint(attr_token_to_name)

df["attributes"] = df["attribute_tokens"].apply(
    lambda tokens: [attr_token_to_name[t] for t in tokens] if isinstance(tokens, list) else []
)

df[["category_name","attributes"]].head()

### Split translation columns

In [ ]:
df[["x","y","z"]]= pd.DataFrame(df["translation"].tolist(), index=df.index)

# df["x"] = df["translation"].apply(lambda t: t[0])
# df["y"] = df["translation"].apply(lambda t: t[1])
# df["z"] = df["translation"].apply(lambda t: t[2])
df[["translation","x","y","z"]].head()

### Split size columns


In [ ]:
df[["width", "length", "height"]] = pd.DataFrame(
    df["size"].tolist(), index=df.index
)

# df["width"] = df["size"].apply(lambda s: s[0])
# df["length"] = df["size"].apply(lambda s: s[1])
# df["height"] = df["size"].apply(lambda s: s[2])
df[["size","width", "length", "height"]].head()

# Derive analysis features
##  Compute box volume


In [ ]:
df["volume"] = df["width"] * df["length"] * df["height"]
df[["volume","width", "length", "height"]].head()

## Extract yaw from quaternion

https://en.wikipedia.org/wiki/Yaw_(rotation) 

https://github.com/nutonomy/nuscenes-devkit/blob/master/docs/schema_nuscenes.md

 "rotation":                <float> [4] -- Coordinate system orientation as quaternion: w, x, y, z.
 


In [ ]:
from pyquaternion import Quaternion

def get_yaw(rotation: list) -> float:
    # Extract the yaw (horizontal heading direction) from the quaternion rotation.
    return Quaternion(rotation).yaw_pitch_roll[0]  # yaw is the first element of the tuple

# Compute yaw for each object in radians.
df["yaw"] = df["rotation"].apply(get_yaw)

# Convert yaw from radians to degrees.
df["yaw_deg"] = np.degrees(df["yaw"])


df[["yaw","yaw_deg"]].head()

## Add simplified class

In [ ]:
nusc_tables["category"]["name"].unique()

In [ ]:
# TODO: update
def simplify_category(cat: str) -> str:
    if cat.startswith("vehicle.car"):
        return "car"
    elif cat.startswith("vehicle.truck"):
        return "truck"
    elif cat.startswith("vehicle.bus"):
        return "bus"
    elif cat.startswith("vehicle.trailer"):
        return "trailer"
    elif cat.startswith("vehicle.construction"):
        return "construction_vehicle"
    elif cat.startswith("vehicle.motorcycle"):
        return "motorcycle"
    elif cat.startswith("vehicle.bicycle"):
        return "bicycle"
    elif cat.startswith("human.pedestrian"):
        return "pedestrian"
    elif cat.startswith("movable_object.trafficcone"):
        return "traffic_cone"
    elif cat.startswith("movable_object.barrier"):
        return "barrier"
    else:
        return "other"
    
df["class_name"] = df["category_name"].apply(simplify_category)
df.head()

## Compute ego-relative distance
Build sample -> ego pose map


In [ ]:
sample_df: pd.DataFrame = nusc_tables["sample"][["token", "data"]].copy()

def get_lidar_top_token(data_dict: dict | None)-> str | None:
    return data_dict.get("LIDAR_TOP") if isinstance(data_dict, dict) else None

sample_df["lidar_sd_token"]= sample_df["data"].apply(get_lidar_top_token)


sample_to_lidar_sd: pd.Series = sample_df.set_index("token")["lidar_sd_token"]

In [ ]:
sample_data_df: pd.DataFrame = nusc_tables["sample_data"][["token", "ego_pose_token"]].copy()
lidar_sd_to_ego_pose: pd.Series = sample_data_df.set_index("token")["ego_pose_token"]

ego_pose_df: pd.DataFrame = nusc_tables["ego_pose"][["token", "translation"]].copy()
ego_pose_to_translation: pd.Series = ego_pose_df.set_index("token")["translation"]

In [ ]:
df["lidar_sd_token"] = df["sample_token"].map(sample_to_lidar_sd)
df["ego_pose_token"] = df["lidar_sd_token"].map(lidar_sd_to_ego_pose)
df["ego_translation"] = df["ego_pose_token"].map(ego_pose_to_translation)

#TODO: understand
df[["ego_x", "ego_y", "ego_z"]] = pd.DataFrame(
    df["ego_translation"].tolist(),
    index=df.index
)

df["distance_ego_2d"] = np.sqrt(
    (df["x"] - df["ego_x"])**2 +
    (df["y"] - df["ego_y"])**2
)

df["distance_ego_3d"] = np.sqrt(
    (df["x"] - df["ego_x"])**2 +
    (df["y"] - df["ego_y"])**2 +
    (df["z"] - df["ego_z"])**2
)

df.head()

## Human-readable visibility

In [ ]:
visibility_map = {
    "1": "0-40% visible",
    "2": "40-60% visible",
    "3": "60-80% visible",
    "4": "80-100% visible",
}

df["visibility_token"] = df["visibility_token"].astype(str)
df["visibility_label"] = df["visibility_token"].map(visibility_map)

df[["visibility_token", "visibility_label"]].head()

## Motion-related flags

In [ ]:
# TODO: the logic is wrong
def has_attribute(attr_list: list, keyword: str) -> bool:
    return any(keyword in attr for attr in attr_list)

df["is_moving"] = df["attributes"].apply(lambda x: has_attribute(x, "moving"))
df["is_parked"] = df["attributes"].apply(lambda x: has_attribute(x, "parked"))
df["is_stopped"] = df["attributes"].apply(lambda x: has_attribute(x, "stopped"))
df["is_standing"] = df["attributes"].apply(lambda x: has_attribute(x, "standing"))
df["is_sitting_lying_down"] = df["attributes"].apply(lambda x: has_attribute(x, "sitting_lying_down"))
df.head()

## Distance bins

In [ ]:
dist_bins: list = [0, 20, 40, 60, 80, 1e6]
dist_labels: list = ["0-20m", "20-40m", "40-60m", "60-80m", "80m+"]

df["dist_bin"] = pd.cut(
    df["distance_ego_2d"],
    bins=dist_bins,
    labels=dist_labels,
    right=False
)

df.head()

## Difficulty flags

In [ ]:
df["lidar_sparse"] = df["num_lidar_pts"] < 5
df["radar_supported"] = df["num_radar_pts"] > 0
df["far_object"] = df["distance_ego_2d"] >= 40
df["low_visibility"] = df["visibility_token"].isin(["1", "2"])

df[["category_name", "class_name", "lidar_sparse", "radar_supported", "far_object","low_visibility"]].head()

## Final DataFrame 
### Quick check

In [ ]:
cols_to_show = [
    "token",
    "class_name",
    "category_name",
    "distance_ego_2d",
    "num_lidar_pts",
    "num_radar_pts",
    "width",
    "length",
    "height",
    "volume",
    "yaw_deg",
    "visibility_label",
    "attributes",
    "is_moving",
    "dist_bin",
    "lidar_sparse",
    "far_object",
]

df[cols_to_show].head()

In [ ]:
eda_cols = [
    "token",
    "sample_token",
    "instance_token",
    "scene_token",
    "scene_name",
    "location",
    "timestamp",
    "category_name",
    "class_name",
    "visibility_token",
    "visibility_label",
    "attributes",
    "x", "y", "z",
    "ego_x", "ego_y", "ego_z",
    "width", "length", "height",
    "volume",
    "yaw", "yaw_deg",
    "distance_ego_2d",
    "distance_ego_3d",
    "num_lidar_pts",
    "num_radar_pts",
    "dist_bin",
    "is_moving",
    "is_parked",
    "is_stopped",
    "lidar_sparse",
    "radar_supported",
    "far_object",
    "low_visibility",
]

eda_df = df[eda_cols].copy()
eda_df.head()

In [ ]:
print(eda_df.shape)
display(eda_df.head())
display(eda_df.isnull().sum().sort_values(ascending=False).head(15))
# display(eda_df.describe(include="all").T.head(20))

# EDA

* when is LiDAR enough, and when does camera or radar add missing information?


* Which object classes are worth focusing on?

* At what distances does LiDAR start failing?

* When could camera help LiDAR?

* When could radar help LiDAR?

* Which scenes/weather/locations are difficult?

* Is the dataset imbalanced in a way that should affect training or evaluation?

* What prediction target is realistic: all 10 classes, or a smaller subset first?



What to check:

missing values

strange distances

zero LiDAR / radar points

class distribution looks reasonable

## Class distribution

In [ ]:
# class_counts:pd.Series = eda_df["class_name"].value_counts().sort_values(ascending=False)
class_counts:pd.Series = eda_df["category_name"].value_counts().sort_values(ascending=False)

plt.figure(figsize=(15, 5))
class_counts.plot(kind="bar")
plt.title("Count per category")
plt.xlabel("Category")
plt.ylabel("Count")
plt.xticks(rotation=90)
plt.show()

In [ ]:
((100 * class_counts / class_counts.sum()).round(2)).astype(str) + "%"


Interpretation:

identify dominant classes

identify rare classes

note that rare classes may have less stable results later

## Distance distribution

In [ ]:
plt.figure(figsize=(8, 5))
eda_df["distance_ego_2d"].hist(bins=50)
plt.title("Object Distance from Ego Vehicle")
plt.xlabel("Distance (m)")
plt.ylabel("Count")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 5))
eda_df.boxplot(column="distance_ego_2d", by="category_name", rot=90, ax=ax)

ax.set_title("Distance by Class")
plt.suptitle("")
ax.set_xlabel("Class")
ax.set_ylabel("Distance (m)")
plt.show()


In [ ]:
# pd.crosstab(eda_df["class_name"], eda_df["dist_bin"])
pd.crosstab(eda_df["category_name"], eda_df["dist_bin"])

Interpretation:

which classes are mostly near-range

which classes appear more often far away

where LiDAR sparsity is likely to increase

## LiDAR
### points per box

In [ ]:
plt.figure(figsize=(8, 5))
eda_df["num_lidar_pts"].hist(bins=50)
plt.title("LiDAR Points per Ground-Truth Box")
plt.xlabel("Number of LiDAR Points")
plt.ylabel("Count")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 6))
# eda_df.boxplot(column="num_lidar_pts", by="class_name", rot=45)
eda_df.boxplot(column="num_lidar_pts", by="category_name", rot=90, ax=ax)

ax.set_title("LiDAR Points per Box by Class")
plt.suptitle("")
ax.set_xlabel("Class")
ax.set_ylabel("LiDAR Points")
plt.show()

In [ ]:
eda_df.head()

In [ ]:
eda_df.groupby("class_name")["num_lidar_pts"].median().sort_values()

Interpretation:

small classes should have fewer points

sparse classes are good candidates for camera fusion

### points vs distance

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(
    eda_df["distance_ego_2d"],
    eda_df["num_lidar_pts"],
    s=5,
    alpha=0.25
)
plt.title("LiDAR Points per Box vs Distance")
plt.xlabel("Distance (m)")
plt.ylabel("LiDAR Points")
plt.show()

In [ ]:
lidar_by_dist = (
    eda_df.groupby("dist_bin", observed=False)["num_lidar_pts"]
    .median()
)

plt.figure(figsize=(8, 5))
lidar_by_dist.plot(marker="o")
plt.title("Median LiDAR Points by Distance Bin")
plt.xlabel("Distance Bin")
plt.ylabel("Median LiDAR Points")
plt.show()

In [ ]:
lidar_class_dist = (
    eda_df.groupby(["class_name", "dist_bin"], observed=False)["num_lidar_pts"]
    .median()
    .unstack()
)

lidar_class_dist

Interpretation:

this is one of the main plots for your thesis

farther objects should show lower LiDAR support

motivates camera fusion especially for sparse classes

## Radar

Add radar only when motion or range is the bottleneck

So the strategic rule is:

radar fusion should mainly help on moving objects, distant objects, or cases where LiDAR geometry alone is weak but radar still gives useful returns.

### points per box


In [ ]:
plt.figure(figsize=(8, 5))
eda_df["num_radar_pts"].hist(bins=50)
plt.title("Radar Points per Ground-Truth Box")
plt.xlabel("Number of radar Points")
plt.ylabel("Count")
plt.show()

### points vs distance

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(
    eda_df["distance_ego_2d"],
    eda_df["num_radar_pts"],
    s=5,
    alpha=0.25
)
plt.title("Radar Points per Box vs Distance")
plt.xlabel("Distance (m)")
plt.ylabel("Radar Points")
plt.show()

In [ ]:
radar_by_dist = (
    eda_df.groupby("dist_bin", observed=False)["num_radar_pts"]
    .mean()
)

plt.figure(figsize=(8, 5))
radar_by_dist.plot(marker="o")
plt.title("Median radar Points by Distance Bin")
plt.xlabel("Distance Bin")
plt.ylabel("Median radar Points")
plt.show()

In [ ]:
eda_df["num_radar_pts"].describe()


## Camera

* Camera helps when the object is visible in the image but poorly represented in the point cloud.
* camera fusion should mainly recover semantic detail and visual evidence that LiDAR misses.

In other words:

LiDAR tells you where and roughly what shape

camera helps tell you what it looks like


## Object size analysis

In [ ]:
plt.figure(figsize=(12, 6))
eda_df.boxplot(column="volume", by="class_name", rot=45)
plt.title("Bounding Box Volume by Class")
plt.suptitle("")
plt.xlabel("Class")
plt.ylabel("Volume")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(
    eda_df["volume"],
    eda_df["num_lidar_pts"],
    s=10,
    alpha=0.25
)
plt.title("Box Volume vs LiDAR Points")
plt.xlabel("Volume")
plt.ylabel("LiDAR Points")
plt.show()

Interpretation:

smaller objects usually have fewer LiDAR returns

useful for explaining why pedestrians and bicycles are harder

## Visibility and occlusion

In [ ]:
eda_df["visibility_label"].value_counts().sort_index()

In [ ]:
plt.figure(figsize=(8, 5))
eda_df["visibility_label"].value_counts().sort_index().plot(kind="bar")
plt.title("Visibility Levels")
plt.xlabel("Visibility")
plt.ylabel("Count")
plt.xticks(rotation=90)
plt.show()

In [ ]:
vis_by_class = pd.crosstab(
    eda_df["class_name"],
    eda_df["visibility_label"],
    normalize="index"
)

vis_by_class.plot(kind="bar", stacked=True, figsize=(12, 6))
plt.title("Visibility Distribution by Class")
plt.xlabel("Class")
plt.ylabel("Proportion")
plt.legend(title="Visibility", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

Interpretation:

classes with more low-visibility examples are harder

useful for later subgroup evaluation

## Motion and radar relevance

In [ ]:
## TODO: why bicyle is never moving?
moving_rate = eda_df.groupby("class_name")["is_moving"].mean().sort_values(ascending=False)

plt.figure(figsize=(10, 5))
moving_rate.plot(kind="bar")
plt.title("Moving Fraction by Class")
plt.xlabel("Class")
plt.ylabel("Fraction Moving")
plt.xticks(rotation=45)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
eda_df["num_radar_pts"].hist(bins=40)
plt.title("Radar Points per Ground-Truth Box")
plt.xlabel("Number of Radar Points")
plt.ylabel("Count")
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
eda_df.boxplot(column="num_radar_pts", by="class_name", rot=45)
plt.title("Radar Points per Box by Class")
plt.suptitle("")
plt.xlabel("Class")
plt.ylabel("Radar Points")
plt.show()

In [ ]:
eda_df.groupby("class_name")["radar_supported"].mean().sort_values(ascending=False)

Interpretation:

moving classes are most relevant for radar fusion

radar is sparse, but support may still be informative

## Domain split: location

In [ ]:
pd.crosstab(eda_df["location"], eda_df["class_name"])

In [ ]:
pd.crosstab(eda_df["location"], eda_df["class_name"]).plot(
    kind="bar",
    stacked=True,
    figsize=(12, 6)
)
plt.title("Class Distribution by Location")
plt.xlabel("Location")
plt.ylabel("Count")
plt.legend(title="Class", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
eda_df.boxplot(column="distance_ego_2d", by="location", rot=90)
plt.title("Distance by Location")
plt.suptitle("")
plt.xlabel("Location")
plt.ylabel("Distance (m)")
plt.show()

Interpretation:

check whether class mix and difficulty differ by location

useful for discussion of domain shift

## Scene-level difficulty

In [ ]:
scene_df = (
    eda_df.groupby(["scene_token", "scene_name", "location"])
    .agg(
        num_annotations=("token", "count"),
        median_distance=("distance_ego_2d", "median"),
        median_lidar_pts=("num_lidar_pts", "median"),
        median_radar_pts=("num_radar_pts", "mean"),
        moving_fraction=("is_moving", "mean"),
        low_visibility_fraction=("low_visibility", "mean"),
    )
    .reset_index()
)

scene_df.head()

In [ ]:
plt.figure(figsize=(8, 5))
scene_df["num_annotations"].hist(bins=30)
plt.title("Annotations per Scene")
plt.xlabel("Number of Annotations")
plt.ylabel("Count")
plt.show()

In [ ]:
scene_df.sort_values("num_annotations", ascending=False).head(10)

Interpretation:

crowded scenes may be more challenging

scenes with low median LiDAR points are good candidates for fusion-focused examples

## Class summary table

In [ ]:
summary_table = (
    eda_df.groupby("class_name")
    .agg(
        count=("token", "count"),
        median_distance=("distance_ego_2d", "median"),
        median_lidar_pts=("num_lidar_pts", "median"),
        median_radar_pts=("num_radar_pts", "median"),
        median_volume=("volume", "median"),
        moving_fraction=("is_moving", "mean"),
        low_visibility_fraction=("low_visibility", "mean"),
        sparse_fraction=("lidar_sparse", "mean"),
        far_fraction=("far_object", "mean"),
        radar_supported_fraction=("radar_supported", "mean"),
    )
    .sort_values("count", ascending=False)
)

summary_table

## Difficulty buckets

In [ ]:
difficulty_table = (
    eda_df.groupby("class_name")
    .agg(
        sparse_fraction=("lidar_sparse", "mean"),
        far_fraction=("far_object", "mean"),
        low_visibility_fraction=("low_visibility", "mean"),
        moving_fraction=("is_moving", "mean"),
        radar_supported_fraction=("radar_supported", "mean"),
    )
    .sort_values("sparse_fraction", ascending=False)
)

difficulty_table

Interpretation:

high sparse_fraction → strong case for camera fusion

high moving_fraction + radar support → strong case for radar fusion

high low_visibility_fraction → difficult subset for all models

## Implications of EDA for Model Design and Evaluation

* early fusion vs late fusion
* feature extraction with mnet


In nuScenes, fusion is mostly an alignment problem

This is the most important practical idea.

The hard part is often not “having multiple sensors.”
The hard part is:

how to align information from different sensors into a shared representation.

In nuScenes, sensors differ in:

* viewpoint

* sparsity

* field of view

* timestamp proximity

* physical meaning of measurement

The devkit schema also makes this clear: sensors are tied together through calibrated sensor and ego-pose transforms, with extrinsics defined relative to the ego vehicle frame.

So basic fusion strategy becomes:

bring all signals into a common frame

fuse only information that is spatially and semantically aligned

avoid naive fusion that mixes noise across modalities

That is why many methods prefer BEV fusion or proposal-level fusion:

BEV is a natural shared space for driving scenes

proposal-level fusion reduces false associations

image-space fusion alone struggles with 3D consistency

# EDA old
where LiDAR is sparse

where objects are far

where visibility is poor

where motion matters

where radar may help

Best way to connect image EDA to your thesis


`Use image EDA to support these claims`:

* Camera helps when LiDAR is sparse but appearance is still informative

* Camera helps with small classes when the object remains visible in image space

* Camera may struggle when projected size is tiny, visibility is poor, or truncation is strong

* Radar can complement both when motion cues exist, even if visual or LiDAR evidence is weak

In [ ]:
#TODO: illustrate visibility, nightime, weather, etc.
#TODO: number of things per frame
#TODO: plot location scene percentage
#TODO: is there class imabalance
#TODO: lidar point by object

## Load tables into DataFrame

In [ ]:
# Ge the list of table names from the nuScenes dataset
NUSC_TABLES: list[str] = nusc.table_names

# NuScenes tables are stored as lists of dicts, but we can easily convert them to pandas DataFrames for easier inspection and analysis.
dfs: dict[str, pd.DataFrame] = {t: pd.DataFrame(getattr(nusc, t)) for t in NUSC_TABLES if hasattr(nusc, t)}
dfs.keys()

In [ ]:
def inspect_df(df: pd.DataFrame, name: str = "df", head: int = 5) -> None:
    rows, cols = df.shape
    print(f"\n--- {name} ---")
    
    # Replace empty strings with NaN for better analysis of missing values.
    df.replace(to_replace="", value=np.nan, inplace=True)  
    
    print(f"DataFrame shape: {rows:,} rows × {cols:,} columns")
    print("\nDataframe columns:", df.columns.tolist())
        
    print("\nConcise summary of a DataFrame:")
    print(df.info(verbose=True))

    display(df.head(head))

# Check the "scene" DataFrame
inspect_df(dfs["scene"], name="scene")

## Class distribution

In [ ]:
sa_df = dfs["sample_annotation"].copy()
sa_df.head()

In [ ]:
# TODO: filter df to plot calls with more than 1000 annotations and below
sa_df['category_name'].value_counts().plot(kind='bar')
plt.title('Annotation count per detection class')
plt.ylabel('Count')
plt.show()

## Annotation dataframe
### Utility functions

In [ ]:
# TODO: make sure to understand the calculation
def get_ego_distances(nusc, sample, ann):
    # 1) Use the sample's top LiDAR frame as the ego reference
    lidar_sd_token = sample['data']['LIDAR_TOP']
    lidar_sd = nusc.get('sample_data', lidar_sd_token)

    # 2) Get ego pose at that LiDAR timestamp
    ego_pose = nusc.get('ego_pose', lidar_sd['ego_pose_token'])

    # 3) Object center from annotation (global coordinates)
    obj_xyz = np.array(ann['translation'], dtype=float)

    # 4) Ego center (global coordinates)
    ego_xyz = np.array(ego_pose['translation'], dtype=float)

    # 5) Relative displacement
    delta = obj_xyz - ego_xyz

    # 6) Distances
    dist_2d = np.linalg.norm(delta[:2])   # x,y only
    dist_3d = np.linalg.norm(delta)       # x,y,z

    return ego_xyz, dist_2d, dist_3d

from pyquaternion import Quaternion



In [ ]:
# might require more explanation about quaternions, and refactoring
def get_yaw(rotation):
    """
    Return the object's heading angle (yaw) from a nuScenes rotation quaternion.
    Yaw means the direction the object is facing in a top-down view.
    Reference:
    https://github.com/nutonomy/nuscenes-devkit/blob/master/python-sdk/nuscenes/nuscenes.py
    """
    quat = Quaternion(rotation)
    yaw, pitch, roll = quat.yaw_pitch_roll
    return yaw

import numpy as np
from pyquaternion import Quaternion

def get_yaw_deg(rotation):
    quat = Quaternion(rotation)
    yaw, pitch, roll = quat.yaw_pitch_roll
    return np.degrees(yaw)

In [ ]:
rows = []

for sample in nusc.sample:
    scene = nusc.get('scene', sample['scene_token'])
    log = nusc.get('log', scene['log_token'])   # contains location info

    ann_tokens = sample['anns']
    for ann_token in ann_tokens:
        ann = nusc.get('sample_annotation', ann_token)

        ego_xyz, dist_2d, dist_3d = get_ego_distances(nusc, sample, ann)

        # Convert attribute tokens to readable attribute names
        attributes = [
            nusc.get('attribute', attr_token)['name']
            for attr_token in ann['attribute_tokens']
        ]

        rows.append({
            'scene_name': scene['name'],
            'scene_token': scene['token'],
            'location': log['location'],

            'sample_token': sample['token'],
            'ann_token': ann['token'],
            'category_name': ann['category_name'],

            'translation_x': ann['translation'][0],
            'translation_y': ann['translation'][1],
            'translation_z': ann['translation'][2],

            'width': ann['size'][0],
            'length': ann['size'][1],
            'height': ann['size'][2],

            'yaw_deg': get_yaw_deg(ann['rotation']),

            'ego_x': ego_xyz[0],
            'ego_y': ego_xyz[1],
            'ego_z': ego_xyz[2],

            'distance_ego_2d': dist_2d,
            'distance_ego_3d': dist_3d,

            'num_lidar_pts': ann['num_lidar_pts'],
            'num_radar_pts': ann['num_radar_pts'],
            'visibility_token': ann['visibility_token'],

            'attributes': attributes
        })

ann_df = pd.DataFrame(rows)
ann_df.head()

In [ ]:
visibility_map = {
    '1': 'v0-40%',
    '2': 'v40-60%',
    '3': 'v60-80%',
    '4': 'v80-100%'
}

ann_df['visibility_label'] = ann_df['visibility_token'].astype(str).map(visibility_map)

## Distance distribution by class
### Global histogram

In [ ]:
ann_df['distance_ego_2d'].hist(bins=50)
plt.title('Object distance from ego vehicle')
plt.xlabel('Distance (m)')
plt.show()

### Box-plot by class

In [ ]:
ann_df.boxplot(column='distance_ego_2d', by='category_name', rot=90)
plt.title('2D Ego Distance by Category')
plt.suptitle('')
plt.xlabel('Category')
plt.ylabel('Distance to ego (m)')
plt.show()

### Distance bins

In [ ]:
bins = [0, 20, 40, 60, 80, 200]
labels = ['0-20', '20-40', '40-60', '60-80', '80+']
ann_df['dist_bin'] = pd.cut(ann_df['distance_ego_2d'], bins=bins, labels=labels, right=False)

pd.crosstab(ann_df['category_name'], ann_df['dist_bin'])

What to write

* which classes are near-range vs long-range

* where sparse LiDAR is likely to appear

### LiDAR points per box
#### Overall

In [ ]:
ann_df['num_lidar_pts'].hist(bins=50)
plt.title('LiDAR points per GT box')
plt.xlabel('Number of LiDAR points')
plt.show()

#### By class

In [ ]:
ann_df.boxplot(column='num_lidar_pts', by='category_name', rot=45)
plt.title('LiDAR points per box by class')
plt.suptitle('')
plt.ylabel('LiDAR points')
plt.show()

In [ ]:
lidar_mean = ann_df.groupby('category_name')['num_lidar_pts'].mean().sort_values(ascending=False)

plt.figure(figsize=(14, 6))
lidar_mean.plot(kind='bar')
plt.title('Average LiDAR Points per Box by Category')
plt.xlabel('Category')
plt.ylabel('Average LiDAR points')
plt.xticks(rotation=45, ha='right')
plt.show()

#### By class and distance

In [ ]:
lidar_stats = ann_df.groupby(['category_name', 'dist_bin'])['num_lidar_pts'].median().unstack()
lidar_stats.plot(kind='bar', figsize=(10, 5))
plt.title('Median LiDAR points per box by class and distance')
plt.ylabel('Median points')
plt.show()

which classes are LiDAR-sparse

how point counts decay with range

which classes camera fusion may help most

### Radar points per box
#### Overall

In [ ]:
ann_df['num_radar_pts'].hist(bins=30)
plt.title('Radar points per GT box')
plt.xlabel('Number of radar points')
plt.show()

#### By class

In [ ]:
ann_df.boxplot(column='num_radar_pts', by='category_name', rot=90)
plt.title('Radar points per box by class')
plt.suptitle('')
plt.ylabel('Radar points')
plt.show()

What to write

many boxes may have few or zero radar points

radar is sparse, but may still be useful for moving objects

### LiDAR points vs distance

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(ann_df['distance_ego_2d'], ann_df['num_lidar_pts'], s=5, alpha=0.3)
plt.xlabel('Distance (m)')
plt.ylabel('LiDAR points')
plt.title('LiDAR points per box vs distance')
plt.show()

In [ ]:
dist_lidar = ann_df.groupby('dist_bin')['num_lidar_pts'].median()
dist_lidar.plot(marker='o')
plt.title('Median LiDAR points by distance bin')
plt.ylabel('Median LiDAR points')
plt.show()

This becomes your main argument for fusion:

farther objects have weaker geometry

small classes become especially hard

### Object size analysis

In [ ]:
ann_df['volume'] = ann_df['width'] * ann_df['length'] * ann_df['height']

In [ ]:
plt.figure(figsize=(24, 6))
ann_df.boxplot(column='volume', by='category_name', rot=90)
plt.title('Bounding box volume by class')
plt.suptitle('')
plt.ylabel('Volume')
plt.show()

In [ ]:
plt.scatter(ann_df['volume'], ann_df['num_lidar_pts'], s=5, alpha=0.3)
plt.xlabel('Box volume')
plt.ylabel('LiDAR points')
plt.title('Box volume vs LiDAR points')
plt.show()

What to write

small objects often have fewer LiDAR returns

visual cues may matter more for small classes

### Visibility / occlusion
#### Plot visibility counts

In [ ]:
ann_df.head()

In [ ]:
ann_df['visibility_label'].value_counts().sort_index().plot(kind='bar')
plt.title('Visibility levels')
plt.xlabel('Visibility token')
plt.ylabel('Count')
plt.show()

#### By class

In [ ]:
vis_class = pd.crosstab(ann_df['category_name'], ann_df['visibility_label'], normalize='index')
vis_class.plot(kind='bar', stacked=True, figsize=(10, 5))
plt.title('Visibility distribution by class')
plt.ylabel('Proportion')
plt.show()

What to write

some classes are more frequently occluded

fusion may help especially in partial visibility cases

### Motion / attributes

In [ ]:
ann_df.head()

In [ ]:
def has_attribute(attr_list, keyword):
    return any(keyword in a for a in attr_list)

ann_df['is_moving'] = ann_df['attributes'].apply(lambda x: has_attribute(x, 'moving'))
ann_df['is_parked'] = ann_df['attributes'].apply(lambda x: has_attribute(x, 'parked'))
ann_df['is_stopped'] = ann_df['attributes'].apply(lambda x: has_attribute(x, 'stopped'))

In [ ]:
moving_rate = ann_df.groupby('category_name')['is_moving'].mean().sort_values(ascending=False)
moving_rate.plot(kind='bar')
plt.title('Moving fraction by class')
plt.ylabel('Fraction moving')
plt.show()

### Orientation / yaw
#### Histogram

In [ ]:
# TODO: evaluate if this is useful
ann_df['yaw_deg'].hist(bins=50)
plt.title('Yaw distribution')
plt.xlabel('Yaw (rad)')
plt.show()

#### By class

In [ ]:
ann_df.head()

In [ ]:
classes = sorted(ann_df['category_name'].dropna().unique())

n_cols = 3
n_rows = (len(classes) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
axes = axes.flatten()

for i, cls in enumerate(classes):
    axes[i].hist(
        ann_df[ann_df['category_name'] == cls]['yaw_deg'].dropna(),
        bins=40,
        range=(-180, 180),
        alpha=0.7
    )
    axes[i].set_title(cls)
    axes[i].set_xlabel('Yaw (degrees)')
    axes[i].set_ylabel('Count')
    axes[i].set_xlim(-180, 180)

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

What to write

which classes are strongly motion-related

why radar could be especially relevant there

### Boston vs Singapore
#### Plot counts by location

In [ ]:
pd.crosstab(ann_df['location'], ann_df['category_name']).plot(
    kind='bar',
    stacked=True,
    figsize=(10, 7)
)

plt.title('Class Distribution by Location')
plt.xlabel('Location')
plt.ylabel('Count')
plt.legend(title='Category', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

#### Compare distance and LiDAR points by location

In [ ]:
ann_df.boxplot(column='distance_ego_2d', by='location', rot=45)
plt.title('Distance by location')
plt.suptitle('')
plt.show()

ann_df.boxplot(column='num_lidar_pts', by='location', rot=45)
plt.title('LiDAR points by location')
plt.suptitle('')
plt.show()

What to write

city-specific differences may affect generalization

scene structure may vary by location

### Scene-level difficulty
#### Scene aggregation

In [ ]:
# TODO: is it useful?
scene_df = ann_df.groupby('scene_token').agg({
    'ann_token': 'count',
    'distance_ego_2d': 'median',
    'num_lidar_pts': 'median',
    'num_radar_pts': 'median',
    'is_moving': 'mean'
}).rename(columns={
    'ann_token': 'num_annotations',
    'distance_ego_2d': 'median_distance',
    'num_lidar_pts': 'median_lidar_pts',
    'num_radar_pts': 'median_radar_pts',
    'is_moving': 'moving_fraction'
}).reset_index()

In [ ]:
scene_df['num_annotations'].hist(bins=30)
plt.title('Objects per scene')
plt.xlabel('Number of annotations')
plt.show()

### Class summary table

In [ ]:
summary_table = ann_df.groupby('category_name').agg(
    count=('ann_token', 'count'),
    median_distance=('distance_ego_2d', 'median'),
    median_lidar_pts=('num_lidar_pts', 'median'),
    median_radar_pts=('num_radar_pts', 'median'),
    median_volume=('volume', 'median'),
    moving_fraction=('is_moving', 'mean')
).sort_values('count', ascending=False)

summary_table

In [ ]:
ann_df['low_visibility'] = ann_df['visibility_token'].isin(['1', '2'])
low_vis = ann_df.groupby('category_name')['low_visibility'].mean()
summary_table['low_visibility_fraction'] = low_vis
summary_table

### Fusion-oriented difficulty buckets

In [ ]:
ann_df['lidar_sparse'] = ann_df['num_lidar_pts'] < 5
ann_df['far_object'] = ann_df['distance_ego_2d'] >= 40
ann_df['radar_supported'] = ann_df['num_radar_pts'] > 0

In [ ]:
difficulty = ann_df.groupby('category_name').agg(
    sparse_fraction=('lidar_sparse', 'mean'),
    far_fraction=('far_object', 'mean'),
    radar_supported_fraction=('radar_supported', 'mean')
)
difficulty

### `Scene`

In [ ]:
def vertical_bar_plot(x: list, y: list, xlabel: str, ylabel: str, title: str) -> None:
    plt.figure(figsize=(16, 6))  
    plt.bar(x, y, color='skyblue')
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.show()
    
vertical_bar_plot(x=dfs["scene"]["name"].to_list(), 
                  y=dfs["scene"]["nbr_samples"].to_list(), 
                  xlabel='scenename', ylabel='number of sample', title='number of samples per scene')

In [ ]:
dfs["scene"]["nbr_samples"].describe()

In [ ]:
for log_token, g in dfs["scene"].groupby("log_token"):
    print(f"log_token: {log_token}, number of scenes: {len(g)}, scene names: {g['name'].tolist()}")

### `Sample`


In [ ]:
# Check the "sample" DataFrame
inspect_df(dfs["sample"], name="sample")

In [ ]:
# TODO: print location of min/max number of annotations per scene
def anns_per_scene(sample_df : pd.DataFrame, scene_df : pd.DataFrame) -> None:

    sample_df: pd.DataFrame = dfs["sample"][["scene_token", "anns"]].copy()
    sample_df["n_anns"] = sample_df["anns"].apply(len)

    # map token -> name
    token_to_name: dict[str, str] = scene_df.set_index("token")["name"].to_dict()

    plt.figure(figsize=(10, 4))
    for scene_token, g in sample_df.groupby("scene_token"):
        label = token_to_name.get(scene_token, scene_token)
        plt.scatter(g.index, g["n_anns"], s=20, alpha=1, label=label)

    plt.xlabel("sample index")
    plt.ylabel("#annotations")
    plt.title("Index vs #annotations per sample (colored by scene)")
    # plt.tight_layout()
    plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", title="scene")
    plt.show()
    
    # How many annotations each sample contains
    print(sample_df["anns"].apply(len).describe())
    
# Plot the number of annotations per sample, colored by scene.
anns_per_scene(sample_df=dfs["sample"], scene_df=dfs["scene"])

In [ ]:
scene_name = "scene-0061"

scene = next(s for s in nusc.scene if s["name"] == scene_name)   # find scene dict
log = nusc.get("log", scene["log_token"])                        # get log record
location = log["location"]                                       # e.g. "boston-seaport"

print(location)

*Notes:*

- **Scene-to-scene variability:** some scenes are **much denser** (more objects per frame) than others.  
- **Outlier scene:** one scene reaches **~150+ objects/frame**, which can **skew averages**.  
- **Within-scene trends:** counts often rise/fall over time (likely entering/leaving **busy areas** like intersections).

**Model implications**
- Evaluate/report **per scene** (or “busy vs light” frames), not only global metrics.  
- Consider **balanced sampling** so one dense scene doesn’t dominate training.

### `Sample data`

In [ ]:
# Check the "sample_data" DataFrame
inspect_df(dfs["sample_data"], "sample_data")

In [ ]:
# cameras should have real dimensions
# radar/lidar will usually be zero
print("unique file formats in sample_data:", dfs["sample_data"]["fileformat"].unique())
print("sample modalities in sample_data:", dfs["sample_data"]["sensor_modality"].unique())
print("channels in sample_data:", dfs["sample_data"]["channel"].unique())


In [ ]:
def count_true_keyframes(df: pd.DataFrame) -> None:
    n_true = df["is_key_frame"].sum()          # booleans sum to counts
    n_total = len(df)
    print("number of True keyframes:", n_true)
    print("total sample_data rows:", n_total)
    print("Percentage of true keyframes: {:.2f}%".format(100 * n_true / n_total))

count_true_keyframes(dfs["sample_data"])

**Note**: 

Most sample_data rows are sweeps, not labeled keyframes.
So for my model:

* Baseline detection model: train/evaluate on keyframes only (clean alignment with labels).

* If I want better performance: add multi-sweep LiDAR/RADAR as extra temporal context around each keyframe.

* EDA / metrics: always separate keyframes vs sweeps, otherwise counts and distributions won’t reflect the labeled training data.

In [ ]:
sd = dfs["sample_data"]

sd["sensor_modality"].value_counts()
sd["channel"].value_counts()
sd.groupby(["sensor_modality", "channel"]).size().sort_values(ascending=False)

### `Sample annotation`

In [ ]:
# Check the "sample_annotation" DataFrame
inspect_df(dfs["sample_annotation"], name="sample_annotation")

In [ ]:
def horizontal_bar_plot(labels, values, xlabel: str, ylabel: str, title: str) -> None:
    plt.figure(figsize=(12, 6))
    plt.barh(labels, values)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    # plt.gca().invert_yaxis()  # optional: largest on top
    # plt.tight_layout()
    plt.show()

horizontal_bar_plot(
    labels=dfs["sample_annotation"]["category_name"].value_counts().index,          # category names
    values=dfs["sample_annotation"]["category_name"].value_counts().values,         # counts
    xlabel="Count by category_name",
    ylabel="category_name",
    title="Distribution of category name in sample_annotation table"
)

**NOTE:**

The plot shows how often each category name appears in the sample annotation table, measured as the number of three dimensional bounding box annotations per category. The distribution is highly imbalanced: a small number of categories, such as `vehicle.car`r and `human.pedestrian.adult`, account for most annotations, while several categories appear only a few times.

This imbalance can lead a detector to focus on frequent categories and perform poorly on rare categories, with low detection rates and unreliable evaluation scores for those rare categories. It can also make overall performance look better than it is, because frequent categories dominate the metrics. Possible solutions include sampling more training frames that contain rare categories, giving higher loss weight to rare categories, using a loss function that reduces the influence of very easy examples, and applying targeted data augmentation for rare categories. When reporting results, it is important to show performance for each category, not only an overall average.

In [ ]:
sa_df = dfs["sample_annotation"].copy()
sa_df.head()

In [ ]:

# sa_df["attribute_name"]= dfs["attribute"].apply(lambda tokens: [nusc.get("attribute", t)["name"] for t in tokens] if isinstance(tokens, list) else [])
# dfs["attribute"].index("token")

token_to_attribute: dict[str, str] = dfs["attribute"].set_index("token")["name"].to_dict()
sa_df["attribute_name"] = sa_df["attribute_tokens"].apply(
    lambda tokens: token_to_attribute.get((tokens or [None])[0])
)
sa_df[["category_name", "attribute_name"]].head()

# sa_df.groupby(["attribute_name","category_name"]).size().sort_values(ascending=False)

In [ ]:
tab = pd.crosstab(
    index=sa_df["category_name"],
    columns=sa_df["attribute_name"],
    rownames=["category_name"],
    # colnames=["attribute_name"],
    dropna=True,
    normalize=False,
    margins=False,
    margins_name="All",
)
tab

In [ ]:
import matplotlib.pyplot as plt

# sort categories by total count (small -> large)
tab_sorted = tab.loc[tab.sum(axis=1).sort_values().index]

tab_sorted.plot(kind="barh", stacked=True, figsize=(12, 5))

plt.xlabel("count")
plt.ylabel("category name")
plt.title("Attribute counts by category (stacked bar, sorted)")
plt.tight_layout()
plt.show()

In [ ]:
my_dict = {"a": [1,23,24], "b": [2,5,6], "c": [4,7,8]}
my_df = pd.DataFrame(my_dict)
print(my_df.head())
my_df.set_index("a", inplace=True)  
print(my_df.head())

### `Instance`


In [ ]:
# Check the "instance" DataFrame
inspect_df(dfs["instance"], name="instance")

### `category`


In [ ]:
# Check the "category" DataFrame
inspect_df(dfs["category"], name="category")

### `Attribute`

In [ ]:
# Check the "attribute" DataFrame
inspect_df(dfs["attribute"], name="attribute")

### `visibility`

In [ ]:
# Check the "visibility" DataFrame
inspect_df(dfs["visibility"], name="visibility")

### `sensor`

In [ ]:
# Check the "sensor" DataFrame
inspect_df(dfs["sensor"], name="sensor")

### `calibrated_sensor`

In [ ]:
# Check the "calibrated_sensor" DataFrame
inspect_df(dfs["calibrated_sensor"], name="calibrated_sensor")

### `ego_pose`

In [ ]:
# Check the "ego_pose" DataFrame
inspect_df(dfs["ego_pose"], name="ego_pose")

### `log`

In [ ]:
# Check the "log" DataFrame
inspect_df(dfs["log"], name="log")

## Planned Additional Exploratory Analyses

# Machine learning

## Machine Learning Research Question
In this Jupyter notebook, I build a supervised 3D object detection pipeline on nuScenes by preparing inputs and targets directly from the official tables. For each sample (one timestamp in the sample table), I assemble the model input by collecting the linked sensor records in sample_data: the LiDAR point cloud file (LIDAR_TOP), the six camera image files (CAM_), and optionally the radar point cloud files (RADAR_). I use calibrated_sensor and ego_pose to transform all sensor measurements into a consistent coordinate frame (ego or global) and to align modalities spatially and temporally. I prepare the ground-truth output for each sample from sample_annotation by extracting category_name and the 3D bounding box parameters (translation, size, rotation) and the object velocity, forming a per-sample set of labeled 3D boxes. With these prepared inputs and outputs, I run a controlled ablation where the only change is which sample_data modalities are included (LiDAR-only, camera+LiDAR, camera+LiDAR+radar) and evaluate each variant using the official nuScenes metrics NDS, mAP, and AVE, followed by a short error breakdown by object class and distance.

## Planned Models and Rationale

## Detailed Machine Learning Strategy

# Additional information